# Tests de mini-funcionalidades de IO trips

Este notebook se usa para probar helpers y bloques internos de `write_trips()` y `read_trips()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados

## Bloque 0. Preparación

### 0.1 Imports generales

Qué prepara: imports básicos, detección de pyarrow y utilidades de filesystem para los tests con Parquet.

In [2]:
import copy
import json
import shutil
import tempfile
from pathlib import Path

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import types as patypes
HAS_PYARROW = True

### 0.2 Imports del módulo

Qué prepara: imports de clases y helpers reales de pylondrina.io.trips.

In [3]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.datasets import TripDataset
from pylondrina.reports import Issue
from pylondrina.errors import ExportError, ValidationError

from pylondrina.io.trips import (
    WriteTripsOptions,
    ReadTripsOptions,
    _validate_write_contract,
    _resolve_write_identity_and_sidecar,
    _create_trips_staging_dir,
    _collect_parquet_categorical_fields,
    _prepare_trips_df_for_parquet_write,
    _write_trips_table_to_staging,
    _write_sidecar_json,
    _assert_staging_complete,
    _commit_staged_trips_artifact,
    _cleanup_staging_dir,
    _validate_read_layout,
    _load_sidecar_json,
    _extract_storage_format,
    _resolve_read_schema_state,
    _read_trips_table_from_storage,
    _finalize_loaded_metadata_state,
    _build_write_trips_summary,
    _build_read_trips_summary,
    _resolve_trips_artifact_paths,
    _trip_schema_to_snapshot,
    _trip_schema_effective_to_snapshot,
)

### 0.3 Helpers de apoyo para test

Qué prepara: utilidades pequeñas para assertions y mensajes, al estilo de tu notebook de OP-02.

In [4]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def assert_counts_by_level(issues, *, errors=None, warnings=None, info=None):
    counts = {"error": 0, "warning": 0, "info": 0}
    for issue in issues:
        counts[issue.level] = counts.get(issue.level, 0) + 1

    if errors is not None:
        assert counts["error"] == errors, f"errors esperado={errors}, actual={counts['error']}"
    if warnings is not None:
        assert counts["warning"] == warnings, f"warnings esperado={warnings}, actual={counts['warning']}"
    if info is not None:
        assert counts["info"] == info, f"info esperado={info}, actual={counts['info']}"


def require_pyarrow():
    assert HAS_PYARROW, (
        "Estos tests requieren pyarrow porque la implementación de write/read "
        "fuerza engine='pyarrow'. Instala pyarrow antes de ejecutar este bloque."
    )

### 0.4 Configuración visual

Qué prepara: display más cómodo.

In [5]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
print("HAS_PYARROW =", HAS_PYARROW)
show_ok("Sección 0 cargada")

Imports OK
HAS_PYARROW = True
OK - Sección 0 cargada


## Bloque 1. Fixtures reutilizables mínimas

Qué prepara: factories pequeñas para schema, schema_effective, dataframe, TripDataset y artefactos persistidos.

In [6]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema(fields: list[FieldSpec], *, version: str = "1.1") -> TripSchema:
    return TripSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=[f.name for f in fields if f.required],
        semantic_rules=None,
    )


def make_trip_schema_effective(
    *,
    dtype_effective: dict | None = None,
    overrides: dict | None = None,
    domains_effective: dict | None = None,
    temporal: dict | None = None,
    fields_effective: list | None = None,
) -> TripSchemaEffective:
    return TripSchemaEffective(
        dtype_effective=dtype_effective or {},
        overrides=overrides or {},
        domains_effective=domains_effective or {},
        temporal=temporal or {},
        fields_effective=fields_effective or [],
    )


def make_trip_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "movement_id": ["m1", "m2", "m3"],
            "trip_id": ["t1", "t2", "t3"],
            "movement_seq": [0, 0, 0],
            "user_id": ["u1", "u2", "u3"],
            "origin_latitude": [-33.45, -33.46, -33.47],
            "origin_longitude": [-70.66, -70.67, -70.68],
            "destination_latitude": [-33.41, -33.42, -33.43],
            "destination_longitude": [-70.61, -70.62, -70.63],
            "mode": ["bus", "metro", "bus"],
            "purpose": ["work", "study", "work"],
            "comment": ["a", "b", "c"],
            "trip_weight": [1.0, 2.5, 1.2],
        }
    )


def make_trip_schema_minimal() -> TripSchema:
    return make_trip_schema(
        [
            make_field("movement_id", "string", required=True),
            make_field("trip_id", "string", required=True),
            make_field("movement_seq", "int", required=True),
            make_field("user_id", "string", required=True),
            make_field("origin_latitude", "float", required=True),
            make_field("origin_longitude", "float", required=True),
            make_field("destination_latitude", "float", required=True),
            make_field("destination_longitude", "float", required=True),
            make_field(
                "mode",
                "categorical",
                required=False,
                domain=DomainSpec(values=["bus", "metro", "walk", "car"], extendable=True),
            ),
            make_field(
                "purpose",
                "categorical",
                required=False,
                domain=DomainSpec(values=["work", "study", "health"], extendable=True),
            ),
            make_field("comment", "string", required=False),
            make_field("trip_weight", "float", required=False),
        ]
    )


def make_trip_schema_effective_minimal() -> TripSchemaEffective:
    return make_trip_schema_effective(
        dtype_effective={
            "mode": "categorical",
            "purpose": "categorical",
            "trip_weight": "float",
        },
        domains_effective={
            "mode": {"values": ["bus", "metro", "walk", "car"]},
            "purpose": {"values": ["work", "study", "health"]},
        },
        temporal={"tier": "tier_3"},
        fields_effective=[
            "movement_id",
            "trip_id",
            "movement_seq",
            "user_id",
            "origin_latitude",
            "origin_longitude",
            "destination_latitude",
            "destination_longitude",
            "mode",
            "purpose",
            "comment",
            "trip_weight",
        ],
    )


def make_tripdataset(
    *,
    validated: bool = True,
    include_dataset_id: bool = True,
    include_artifact_id: bool = False,
) -> TripDataset:
    schema = make_trip_schema_minimal()
    schema_effective = make_trip_schema_effective_minimal()
    metadata = {
        "is_validated": validated,
        "events": [],
        "mappings": {
            "field_correspondence": {
                "movement_id": "movement_id_src",
                "mode": "mode_src",
            },
            "value_correspondence": {
                "mode": {
                    "micro": "bus",
                    "subte": "metro",
                }
            },
        },
        "domains_effective": copy.deepcopy(schema_effective.domains_effective),
        "temporal": {"tier": "tier_3"},
    }
    if include_dataset_id:
        metadata["dataset_id"] = "dset_test_001"
    if include_artifact_id:
        metadata["artifact_id"] = "art_test_001"

    provenance = {
        "source": {"name": "synthetic", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
    }

    return TripDataset(
        data=make_trip_df(),
        schema=schema,
        schema_version=schema.version,
        provenance=provenance,
        field_correspondence={"movement_id": "movement_id_src", "mode": "mode_src"},
        value_correspondence={"mode": {"micro": "bus", "subte": "metro"}},
        metadata=metadata,
        schema_effective=schema_effective,
    )


def make_sidecar_payload(
    *,
    schema: TripSchema | None = None,
    schema_effective: TripSchemaEffective | None = None,
    metadata: dict | None = None,
    provenance: dict | None = None,
    dataset_id: str = "dset_sidecar_001",
    artifact_id: str = "art_sidecar_001",
    compression: str | None = "snappy",
    storage_format: str = "parquet",
) -> dict:
    schema = schema or make_trip_schema_minimal()
    schema_effective = schema_effective or make_trip_schema_effective_minimal()
    metadata = copy.deepcopy(metadata) if metadata is not None else {
        "dataset_id": dataset_id,
        "artifact_id": artifact_id,
        "is_validated": True,
        "events": [],
        "mappings": {
            "field_correspondence": {"movement_id": "movement_id_src"},
            "value_correspondence": {"mode": {"micro": "bus"}},
        },
        "domains_effective": copy.deepcopy(schema_effective.domains_effective),
    }
    provenance = copy.deepcopy(provenance) if provenance is not None else {
        "source": {"name": "synthetic", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
    }

    return {
        "dataset_type": "trips",
        "format": "golondrina",
        "layout_version": "1.1",
        "storage": {
            "format": storage_format,
            "options": {"compression": compression},
        },
        "dataset_id": dataset_id,
        "artifact_id": artifact_id,
        "files": {
            "data": "trips.parquet",
            "metadata": "trips.metadata.json",
        },
        "schema": _trip_schema_to_snapshot(schema),
        "schema_effective": _trip_schema_effective_to_snapshot(schema_effective),
        "provenance": provenance,
        "metadata": metadata,
    }


def materialize_minimal_formal_artifact(
    root_dir: Path,
    *,
    df: pd.DataFrame | None = None,
    schema: TripSchema | None = None,
    schema_effective: TripSchemaEffective | None = None,
    metadata: dict | None = None,
    provenance: dict | None = None,
    dataset_id: str = "dset_artifact_001",
    artifact_id: str = "art_artifact_001",
    compression: str | None = "snappy",
) -> tuple:
    """
    Crea en disco un artefacto formal mínimo para tests de lectura.
    Esto es setup de pruebas, no el subject under test.
    """
    require_pyarrow()

    root_dir.mkdir(parents=True, exist_ok=True)
    df = df.copy() if df is not None else make_trip_df()
    schema = schema or make_trip_schema_minimal()
    schema_effective = schema_effective or make_trip_schema_effective_minimal()

    data_path = root_dir / "trips.parquet"
    sidecar_path = root_dir / "trips.metadata.json"

    df.to_parquet(
        data_path,
        index=False,
        compression=None if compression == "none" else compression,
        engine="pyarrow",
    )

    payload = make_sidecar_payload(
        schema=schema,
        schema_effective=schema_effective,
        metadata=metadata,
        provenance=provenance,
        dataset_id=dataset_id,
        artifact_id=artifact_id,
        compression=compression,
    )
    sidecar_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    return _resolve_trips_artifact_paths(root_dir), payload

## Bloque 2. Helpers internos de uso general


### Test 2.1 - _resolve_trips_artifact_paths

Qué prueba: que el layout formal se resuelva siempre a trips.parquet, trips.metadata.json y metadata.json legacy.

In [7]:
root = Path("tmp_demo_artifact")
paths = _resolve_trips_artifact_paths(root)

assert paths.root_dir == root
assert paths.data_path == root / "trips.parquet"
assert paths.sidecar_path == root / "trips.metadata.json"
assert paths.legacy_sidecar_path == root / "metadata.json"

show_ok("Test 2.1 - _resolve_trips_artifact_paths")

OK - Test 2.1 - _resolve_trips_artifact_paths


### Test 2.2 - _create_trips_staging_dir y _cleanup_staging_dir

Qué prueba: creación de staging hermano y cleanup normal.

In [12]:
with tempfile.TemporaryDirectory() as td:
    final_dir = Path(td) / "artifact_final"
    issues = []

    staging_dir = _create_trips_staging_dir(final_dir, issues=issues)

    assert staging_dir.exists()
    assert staging_dir.is_dir()
    assert staging_dir.parent == final_dir.parent
    assert issues == []

    _cleanup_staging_dir(
        staging_dir,
        final_dir,
        ["trips.parquet", "trips.metadata.json"],
        issues,
    )
    assert not staging_dir.exists()

show_ok("Test 2.2 - _create_trips_staging_dir + _cleanup_staging_dir")

OK - Test 2.2 - _create_trips_staging_dir + _cleanup_staging_dir


### Test 2.3 - _write_sidecar_json, _load_sidecar_json y _extract_storage_format

Qué prueba: sidecar válido round-trip y extracción correcta del backend; además, error cuando el formato no es soportado.

In [13]:
with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir(parents=True, exist_ok=True)
    paths = _resolve_trips_artifact_paths(root)

    payload = make_sidecar_payload()
    issues = []

    _write_sidecar_json(
        payload,
        paths.sidecar_path,
        issues=issues,
        destination_path=root,
        dataset_id=payload["dataset_id"],
        artifact_id=payload["artifact_id"],
    )
    assert paths.sidecar_path.exists()
    assert issues == []

    issues_load = []
    loaded = _load_sidecar_json(
        paths.sidecar_path,
        issues=issues_load,
        destination_path=root,
    )
    assert loaded["dataset_type"] == "trips"
    assert loaded["files"]["data"] == "trips.parquet"
    assert loaded["schema"]["version"] == "1.1"
    assert issues_load == []

    issues_fmt = []
    fmt = _extract_storage_format(loaded, strict=False, issues=issues_fmt)
    assert fmt == "parquet"
    assert issues_fmt == []

    bad_payload = copy.deepcopy(loaded)
    bad_payload["storage"]["format"] = "feather"
    issues_bad = []
    try:
        _extract_storage_format(bad_payload, strict=False, issues=issues_bad)
        raise AssertionError("Debió fallar por storage.format no soportado")
    except ExportError:
        assert_issue_present(issues_bad, "READ.STORAGE.UNSUPPORTED_FORMAT")

show_ok("Test 2.3 - sidecar json + _extract_storage_format")

OK - Test 2.3 - sidecar json + _extract_storage_format


### Test 2.4 - _assert_staging_complete

Qué prueba: que el helper acepte staging completo y falle si falta un artefacto.

In [14]:
# Caso completo
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_ok"
    staging_root.mkdir()
    paths = _resolve_trips_artifact_paths(staging_root)
    paths.data_path.touch()
    paths.sidecar_path.touch()

    issues = []
    _assert_staging_complete(
        paths,
        expected_files=["trips.parquet", "trips.metadata.json"],
        issues=issues,
        destination_path=staging_root,
    )
    assert issues == []

# Caso incompleto
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_bad"
    staging_root.mkdir()
    paths = _resolve_trips_artifact_paths(staging_root)
    paths.data_path.touch()

    issues = []
    try:
        _assert_staging_complete(
            paths,
            expected_files=["trips.parquet", "trips.metadata.json"],
            issues=issues,
            destination_path=staging_root,
        )
        raise AssertionError("Debió fallar por staging incompleto")
    except ExportError:
        assert_issue_present(issues, "WRT.IO.STAGING_INCOMPLETE")

show_ok("Test 2.4 - _assert_staging_complete")

OK - Test 2.4 - _assert_staging_complete


## Bloque 3. Mini-funcionalidad crítica de categóricos Parquet

### Test 3.1 - _collect_parquet_categorical_fields

Qué prueba: detección de campos categóricos desde schema, schema_effective.dtype_effective y schema_effective.domains_effective.

In [15]:
df = pd.DataFrame(
    {
        "mode": ["bus", "metro"],
        "purpose": ["work", "study"],
        "gender": ["M", "F"],
        "comment": ["a", "b"],
    }
)

schema = make_trip_schema(
    [
        make_field("mode", "categorical"),
        make_field("purpose", "string"),
        make_field("gender", "string"),
        make_field("comment", "string"),
    ]
)

schema_effective = make_trip_schema_effective(
    dtype_effective={"purpose": "categorical"},
    domains_effective={"gender": {"values": ["M", "F"]}},
)

categorical_fields = _collect_parquet_categorical_fields(df, schema, schema_effective)

assert set(categorical_fields) == {"mode", "purpose", "gender"}
assert "comment" not in categorical_fields

show_ok("Test 3.1 - _collect_parquet_categorical_fields")

OK - Test 3.1 - _collect_parquet_categorical_fields


### Test 3.2 - _prepare_trips_df_for_parquet_write

Qué prueba: 

- detecta categóricos desde schema y schema_effective,
- convierte a category,
- no toca columnas no categóricas,
- remueve categorías no usadas,
- y no muta el dataframe original.

In [16]:
df = pd.DataFrame(
    {
        "mode": pd.Series(
            pd.Categorical(["bus", "bus", "metro"], categories=["bus", "metro", "car"])
        ),
        "purpose": ["work", "study", "work"],
        "comment": ["ok1", "ok2", "ok3"],
        "trip_weight": [1.0, 2.0, 3.0],
    }
)

schema = make_trip_schema(
    [
        make_field("mode", "categorical"),
        make_field("purpose", "string"),
        make_field("comment", "string"),
        make_field("trip_weight", "float"),
    ]
)

schema_effective = make_trip_schema_effective(
    dtype_effective={"purpose": "categorical"},
    domains_effective={"purpose": {"values": ["work", "study", "health"]}},
)

prepared = _prepare_trips_df_for_parquet_write(df, schema, schema_effective)

# mode sigue categórico y elimina categoría no usada "car"
assert isinstance(prepared["mode"].dtype, pd.CategoricalDtype)
assert list(prepared["mode"].cat.categories) == ["bus", "metro"]

# purpose se volvió categórico desde schema_effective
assert isinstance(prepared["purpose"].dtype, pd.CategoricalDtype)
assert list(prepared["purpose"].cat.categories) == ["study", "work"] or list(prepared["purpose"].cat.categories) == ["work", "study"]

# columnas no categóricas no se tocan
assert not isinstance(prepared["comment"].dtype, pd.CategoricalDtype)
assert prepared["comment"].dtype == df["comment"].dtype
assert prepared["trip_weight"].dtype == df["trip_weight"].dtype

# no muta el dataframe original
assert isinstance(df["mode"].dtype, pd.CategoricalDtype)
assert list(df["mode"].cat.categories) == ["bus", "metro", "car"]
assert df["purpose"].dtype == object

show_ok("Test 3.2 - _prepare_trips_df_for_parquet_write")

OK - Test 3.2 - _prepare_trips_df_for_parquet_write


### Test 3.3 - _write_trips_table_to_staging + verificación de dictionary encoding

Qué prueba: escritura física a parquet usando la preparación categórica y, si pyarrow está disponible, verificación de codificación tipo dictionary en el archivo persistido.

In [22]:
require_pyarrow()

df = make_trip_df()
schema = make_trip_schema_minimal()
schema_effective = make_trip_schema_effective_minimal()

with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir(parents=True, exist_ok=True)
    paths = _resolve_trips_artifact_paths(root)

    issues = []
    _write_trips_table_to_staging(
        df,
        paths.data_path,
        storage_format="parquet",
        parquet_compression="snappy",
        schema=schema,
        schema_effective=schema_effective,
        issues=issues,
        destination_path=root,
    )

    assert paths.data_path.exists()
    assert issues == []

    # Lectura básica por el helper de read
    issues_read = []
    df_back = _read_trips_table_from_storage(
        paths.data_path,
        storage_format="parquet",
        issues=issues_read,
        destination_path=root,
    )
    assert len(df_back) == len(df)
    assert list(df_back.columns) == list(df.columns)
    assert issues_read == []

    # Verificación parquet-level de dictionary encoding en columnas categóricas.
    parquet_file = pq.ParquetFile(paths.data_path)
    try:
        names = parquet_file.schema_arrow.names

        for col_name in ["mode", "purpose"]:
            col_idx = names.index(col_name)
            encodings = {
                str(enc).upper()
                for enc in parquet_file.metadata.row_group(0).column(col_idx).encodings
            }
            assert any("DICTIONARY" in enc for enc in encodings), (
                f"{col_name} no quedó con dictionary encoding: {encodings}"
            )
    finally:
        parquet_file.close()

show_ok("Test 3.3 - _write_trips_table_to_staging + dictionary encoding")

OK - Test 3.3 - _write_trips_table_to_staging + dictionary encoding


## Bloque 4. Helpers principales de Write trips

### Test 4.1 - _validate_write_contract happy path

Qué prueba: caso correcto sin excepción.

In [23]:
trips = make_tripdataset(validated=True)

with tempfile.TemporaryDirectory() as td:
    issues = []
    _validate_write_contract(
        trips,
        Path(td) / "artifact",
        WriteTripsOptions(),
        issues=issues,
    )

    assert_issue_absent(issues, "WRT.VALIDATION.REQUIRED_NOT_VALIDATED")
    assert_issue_absent(issues, "WRT.CORE.INVALID_TRIPDATASET")
    assert_issue_absent(issues, "WRT.CORE.INVALID_DATA_SURFACE")

show_ok("Test 4.1 - _validate_write_contract happy path")

OK - Test 4.1 - _validate_write_contract happy path


### Test 4.2 - _validate_write_contract fatal por require_validated=True

Qué prueba: la precondición más importante del write.

In [24]:
trips = make_tripdataset(validated=False)

with tempfile.TemporaryDirectory() as td:
    issues = []
    try:
        _validate_write_contract(
            trips,
            Path(td) / "artifact",
            WriteTripsOptions(require_validated=True),
            issues=issues,
        )
        raise AssertionError("Debió fallar por dataset no validado")
    except ValidationError:
        assert_issue_present(issues, "WRT.VALIDATION.REQUIRED_NOT_VALIDATED")

show_ok("Test 4.2 - _validate_write_contract fatal validated")

OK - Test 4.2 - _validate_write_contract fatal validated


### Test 4.3 - _validate_write_contract info para dataframe vacío

Qué prueba: dataset vacío permitido pero con issue informativo

In [25]:
trips = make_tripdataset(validated=True)
trips.data = trips.data.iloc[0:0].copy()

with tempfile.TemporaryDirectory() as td:
    issues = []
    _validate_write_contract(
        trips,
        Path(td) / "artifact",
        WriteTripsOptions(),
        issues=issues,
    )

    assert_issue_present(issues, "WRT.CORE.EMPTY_DATAFRAME")
    assert_counts_by_level(issues, info=1)

show_ok("Test 4.3 - _validate_write_contract empty dataframe")

OK - Test 4.3 - _validate_write_contract empty dataframe


### Test 4.4 - _resolve_write_identity_and_sidecar

Qué prueba: creación/regeneración de identidad y sidecar completo, incluyendo evento write_trips.

In [26]:
trips = make_tripdataset(validated=True, include_dataset_id=False, include_artifact_id=False)
paths = _resolve_trips_artifact_paths(Path("tmp_artifact_for_resolve"))
resolved = _resolve_write_identity_and_sidecar(
    trips,
    paths,
    WriteTripsOptions(),
    existing_issues=[],
)

assert resolved.dataset_id_status == "created"
assert isinstance(resolved.dataset_id, str) and resolved.dataset_id
assert isinstance(resolved.artifact_id, str) and resolved.artifact_id

assert resolved.sidecar_payload["dataset_type"] == "trips"
assert resolved.sidecar_payload["files"]["data"] == "trips.parquet"
assert "schema" in resolved.sidecar_payload
assert "schema_effective" in resolved.sidecar_payload
assert resolved.metadata_for_persist["dataset_id"] == resolved.dataset_id
assert resolved.metadata_for_persist["artifact_id"] == resolved.artifact_id
assert resolved.metadata_for_persist["events"][-1]["op"] == "write_trips"

assert_issue_present(resolved.issues, "WRT.METADATA.DATASET_ID_CREATED")
assert_json_safe(resolved.sidecar_payload, "sidecar_payload")

show_ok("Test 4.4 - _resolve_write_identity_and_sidecar")

OK - Test 4.4 - _resolve_write_identity_and_sidecar


### Test 4.5 - _commit_staged_trips_artifact success

Qué prueba: commit correcto desde staging al directorio final.

In [27]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    staging = parent / "staging"
    final_dir = parent / "artifact"
    staging.mkdir()

    (staging / "trips.parquet").write_text("dummy parquet placeholder", encoding="utf-8")
    (staging / "trips.metadata.json").write_text("{}", encoding="utf-8")

    issues = []
    _commit_staged_trips_artifact(
        staging,
        final_dir,
        mode="error_if_exists",
        files_written=["trips.parquet", "trips.metadata.json"],
        issues=issues,
    )

    assert final_dir.exists()
    assert not staging.exists()
    assert (final_dir / "trips.parquet").exists()
    assert (final_dir / "trips.metadata.json").exists()
    assert issues == []

show_ok("Test 4.5 - _commit_staged_trips_artifact success")

OK - Test 4.5 - _commit_staged_trips_artifact success


### Test 4.6 - _commit_staged_trips_artifact fatal por colisión

Qué prueba: mode="error_if_exists".

In [28]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    final_dir = parent / "artifact"
    staging = parent / "staging"

    final_dir.mkdir()
    (final_dir / "old.txt").write_text("legacy", encoding="utf-8")

    staging.mkdir()
    (staging / "trips.parquet").write_text("dummy parquet placeholder", encoding="utf-8")
    (staging / "trips.metadata.json").write_text("{}", encoding="utf-8")

    issues = []
    try:
        _commit_staged_trips_artifact(
            staging,
            final_dir,
            mode="error_if_exists",
            files_written=["trips.parquet", "trips.metadata.json"],
            issues=issues,
        )
        raise AssertionError("Debió fallar por destino ya existente")
    except ExportError:
        assert_issue_present(issues, "WRT.DEST.ALREADY_EXISTS")

show_ok("Test 4.6 - _commit_staged_trips_artifact collision")

OK - Test 4.6 - _commit_staged_trips_artifact collision


### Test 4.7 - _build_write_trips_summary

Qué prueba: summary mínimo estable.

In [33]:
summary = _build_write_trips_summary(
    n_rows=3,
    path=Path("/tmp/artifact"),
    artifact_id="art_001",
    dataset_id_status="created",
    dataset_id="dset_001",
    storage_format="parquet",
    files_written=["trips.parquet", "trips.metadata.json"],
)

assert summary["n_rows"] == 3
assert Path(summary["path"]) == Path("/tmp/artifact")
assert summary["artifact_id"] == "art_001"
assert summary["dataset_id"] == "dset_001"
assert summary["dataset_id_status"] == "created"
assert summary["storage_format"] == "parquet"
assert summary["files_written"] == ["trips.parquet", "trips.metadata.json"]
assert_json_safe(summary, "write_summary")

show_ok("Test 4.7 - _build_write_trips_summary")

OK - Test 4.7 - _build_write_trips_summary


## Bloque 5. Helpers principales de Read trips

### Test 5.1 - _validate_read_layout happy path

Qué prueba: layout formal correcto.

In [34]:
require_pyarrow()

with tempfile.TemporaryDirectory() as td:
    paths, payload = materialize_minimal_formal_artifact(Path(td) / "artifact")
    issues = []

    _validate_read_layout(paths.root_dir, paths, issues=issues)

    assert issues == []

show_ok("Test 5.1 - _validate_read_layout happy path")

OK - Test 5.1 - _validate_read_layout happy path


### Test 5.2 - _validate_read_layout fatal por sidecar legacy

Qué prueba: rechazo explícito de metadata.json.

In [35]:
with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir(parents=True, exist_ok=True)

    (root / "trips.parquet").write_text("placeholder", encoding="utf-8")
    (root / "metadata.json").write_text("{}", encoding="utf-8")

    paths = _resolve_trips_artifact_paths(root)
    issues = []
    try:
        _validate_read_layout(root, paths, issues=issues)
        raise AssertionError("Debió fallar por sidecar legacy")
    except ExportError:
        assert_issue_present(issues, "READ.LAYOUT.LEGACY_SIDECAR_DETECTED")

show_ok("Test 5.2 - _validate_read_layout legacy sidecar")

OK - Test 5.2 - _validate_read_layout legacy sidecar


### Test 5.3 - _resolve_read_schema_state con precedencia de options.schema

Qué prueba: que options.schema mande y que el mismatch quede trazado.

In [36]:
schema_metadata = make_trip_schema_minimal()
schema_options = make_trip_schema(
    [
        make_field("movement_id", "string", required=True),
        make_field("trip_id", "string", required=True),
    ],
    version="9.9",
)

payload = make_sidecar_payload(schema=schema_metadata)
state = _resolve_read_schema_state(
    payload,
    ReadTripsOptions(schema=schema_options, strict=False, keep_metadata=True),
)

assert state.schema.version == "9.9"
assert state.schema_source == "options"
assert state.schema_mismatch is True
assert_issue_present(state.issues, "READ.SCHEMA.MISMATCH")

show_ok("Test 5.3 - _resolve_read_schema_state precedence + mismatch")

OK - Test 5.3 - _resolve_read_schema_state precedence + mismatch


### Test 5.4 - _resolve_read_schema_state con schema_effective faltante y recovery

Qué prueba: caso degradado bajo strict=False.

In [37]:
payload = make_sidecar_payload()
payload.pop("schema_effective")

state = _resolve_read_schema_state(
    payload,
    ReadTripsOptions(schema=None, strict=False, keep_metadata=True),
)

assert isinstance(state.schema_effective, TripSchemaEffective)
assert state.schema_effective.to_dict() == TripSchemaEffective().to_dict()
assert_issue_present(state.issues, "READ.SCHEMA_EFFECTIVE.DEFAULTED")

show_ok("Test 5.4 - _resolve_read_schema_state default schema_effective")

OK - Test 5.4 - _resolve_read_schema_state default schema_effective


### Test 5.5 - _resolve_read_schema_state fatal sin schema recuperable

Qué prueba: falla si no hay options.schema ni snapshot usable.

In [38]:
payload = make_sidecar_payload()
payload.pop("schema")

try:
    _resolve_read_schema_state(
        payload,
        ReadTripsOptions(schema=None, strict=False, keep_metadata=True),
    )
    raise AssertionError("Debió fallar por schema no recuperable")
except ExportError as e:
    assert "READ.SCHEMA.UNAVAILABLE" in str(e)

show_ok("Test 5.5 - _resolve_read_schema_state fatal unavailable")

OK - Test 5.5 - _resolve_read_schema_state fatal unavailable


### Test 5.6 - _read_trips_table_from_storage

Qué prueba: lectura correcta de trips.parquet.

In [39]:
require_pyarrow()

with tempfile.TemporaryDirectory() as td:
    paths, payload = materialize_minimal_formal_artifact(Path(td) / "artifact")
    issues = []

    df_loaded = _read_trips_table_from_storage(
        paths.data_path,
        storage_format="parquet",
        issues=issues,
        destination_path=paths.root_dir,
    )

    assert len(df_loaded) == 3
    assert list(df_loaded.columns) == list(make_trip_df().columns)
    assert issues == []

show_ok("Test 5.6 - _read_trips_table_from_storage")

OK - Test 5.6 - _read_trips_table_from_storage


### Test 5.7 - _finalize_loaded_metadata_state normal

Qué prueba: preserva identidad cargada y fuerza is_validated=False

In [40]:
metadata = {
    "dataset_id": "dset_ok",
    "artifact_id": "art_ok",
    "is_validated": True,
    "events": [],
}
sidecar_payload = {
    "dataset_id": "dset_ok",
    "artifact_id": "art_ok",
}

state = _finalize_loaded_metadata_state(
    metadata,
    sidecar_payload=sidecar_payload,
    strict=False,
    destination_path=Path("/tmp/fake_artifact"),
)

assert state.dataset_id == "dset_ok"
assert state.dataset_id_status == "loaded"
assert state.artifact_id == "art_ok"
assert state.artifact_id_status == "loaded"
assert state.metadata["is_validated"] is False
assert_issue_present(state.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

show_ok("Test 5.7 - _finalize_loaded_metadata_state normal")

OK - Test 5.7 - _finalize_loaded_metadata_state normal


### Test 5.8 - _finalize_loaded_metadata_state recovery con strict=False

Qué prueba: regeneración de dataset_id, artifact_id=None y forcing de no validado.

In [41]:
metadata = {
    "is_validated": True,
    "events": [],
}
sidecar_payload = {
    "dataset_id": "",
    "artifact_id": None,
}

state = _finalize_loaded_metadata_state(
    metadata,
    sidecar_payload=sidecar_payload,
    strict=False,
    destination_path=Path("/tmp/fake_artifact"),
)

assert state.dataset_id_status == "regenerated"
assert isinstance(state.dataset_id, str) and state.dataset_id
assert state.artifact_id is None
assert state.artifact_id_status == "missing_or_invalid"
assert state.metadata["is_validated"] is False

assert_issue_present(state.issues, "READ.METADATA.DATASET_ID_REGENERATED")
assert_issue_present(state.issues, "READ.METADATA.ARTIFACT_ID_SET_NONE")
assert_issue_present(state.issues, "READ.METADATA.VALIDATED_FORCED_FALSE")

show_ok("Test 5.8 - _finalize_loaded_metadata_state recovery")

OK - Test 5.8 - _finalize_loaded_metadata_state recovery


### Test 5.9 - _build_read_trips_summary

Qué prueba: summary mínimo estable de read.

In [43]:
summary = _build_read_trips_summary(
    n_rows=3,
    n_columns=12,
    path=Path("/tmp/artifact"),
    storage_format="parquet",
    schema_source="metadata",
    schema_mismatch=False,
    dataset_id_status="loaded",
    dataset_id="dset_001",
    artifact_id_status="loaded",
    artifact_id="art_001",
)

assert summary["n_rows"] == 3
assert summary["n_columns"] == 12
assert Path(summary["path"]) == Path("/tmp/artifact")
assert summary["storage_format"] == "parquet"
assert summary["schema_source"] == "metadata"
assert summary["schema_mismatch"] is False
assert summary["dataset_id"] == "dset_001"
assert summary["artifact_id"] == "art_001"
assert_json_safe(summary, "read_summary")

show_ok("Test 5.9 - _build_read_trips_summary")

OK - Test 5.9 - _build_read_trips_summary


## Bloque 6: Smoke tests integrados de write_trips / read_trips

### Test 6.1 - Setup visible de smoke tests

Qué prepara: un directorio visible dentro de tu carpeta de trabajo para que puedas inspeccionar manualmente los artefactos persistidos (trips.parquet y trips.metadata.json) sin depender de tempfile.

In [46]:
from pathlib import Path
import json
import shutil
import copy
import pandas as pd

require_pyarrow()

from pylondrina.io.trips import (
    write_trips,
    read_trips,
    WriteTripsOptions,
    ReadTripsOptions,
)

SMOKE_ROOT = Path("./tmp_smoke")


def reset_smoke_root() -> Path:
    if SMOKE_ROOT.exists():
        shutil.rmtree(SMOKE_ROOT)
    SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
    return SMOKE_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = SMOKE_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


root = reset_smoke_root()
print("SMOKE_ROOT =", root.resolve())
show_ok("Test 6.1 - setup visible de smoke tests")

SMOKE_ROOT = D:\Memoria\repos\pylondrina\notebooks\testing\io_trips\tmp_smoke
OK - Test 6.1 - setup visible de smoke tests


### Test 6.2 - smoke test de write_trips happy path mínimo

Qué prueba: caso correcto de escritura formal con dataset validado. Verifica:

- OperationReport,
- layout formal,
- side effects en metadata,
- sidecar consistente,
- y summary mínimo.

In [50]:
case_dir = make_case_dir("case_01_write_happy")
artifact_dir = case_dir / "artifact_write_happy"

trips = make_tripdataset(validated=True)
data_before = trips.data.copy(deep=True)
metadata_before = copy.deepcopy(trips.metadata)

report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
    ),
)

assert report.ok is True
assert len(report.issues) == 0

# Layout formal persistido
assert artifact_dir.exists()
assert (artifact_dir / "trips.parquet").exists()
assert (artifact_dir / "trips.metadata.json").exists()

# Summary y parameters
assert report.summary["n_rows"] == len(trips.data)
assert Path(report.summary["path"]) == artifact_dir
assert report.summary["artifact_id"] == trips.metadata["artifact_id"]
assert report.parameters["storage_format"] == "parquet"
assert report.parameters["mode"] == "error_if_exists"

# Side effects en memoria
assert "dataset_id" in trips.metadata
assert "artifact_id" in trips.metadata
assert trips.metadata["is_validated"] is True
assert len(trips.metadata["events"]) == len(metadata_before["events"]) + 1
assert trips.metadata["events"][-1]["op"] == "write_trips"

# El dataframe no debe mutar
pd.testing.assert_frame_equal(trips.data, data_before)

# Sidecar cargable e íntegro
sidecar = json.loads((artifact_dir / "trips.metadata.json").read_text(encoding="utf-8"))
assert sidecar["dataset_type"] == "trips"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"
assert sidecar["files"]["data"] == "trips.parquet"
assert sidecar["files"]["metadata"] == "trips.metadata.json"
assert sidecar["dataset_id"] == trips.metadata["dataset_id"]
assert sidecar["artifact_id"] == trips.metadata["artifact_id"]
assert "schema" in sidecar
assert "schema_effective" in sidecar
assert "metadata" in sidecar
assert "provenance" in sidecar

display(report)
show_ok("Test 6.2 - write_trips happy path mínimo")

OperationReport(ok=True, issues=[], summary={'n_rows': 3, 'files_written': ['trips.parquet', 'trips.metadata.json'], 'path': 'tmp_smoke\\case_01_write_happy\\artifact_write_happy', 'dataset_id': 'dset_test_001', 'artifact_id': 'art_accf9e31-ff27-4d70-aa4b-0a02d06a0ffc', 'dataset_id_status': 'preserved', 'storage_format': 'parquet'}, parameters={'path': 'tmp_smoke\\case_01_write_happy\\artifact_write_happy', 'mode': 'error_if_exists', 'require_validated': True, 'storage_format': 'parquet', 'parquet_compression': 'snappy'})

OK - Test 6.2 - write_trips happy path mínimo


### Test 6.3 - smoke test de categóricos persistidos eficientemente

Qué prueba: que el artefacto escrito por write_trips realmente deja evidencias de codificación tipo dictionary encoding en columnas categóricas, y además da una forma visible de inspección.

Aquí hago dos verificaciones:

1. Verificación fuerte: inspección Parquet-level de encodings en mode y purpose.
2. Verificación orientativa: comparación de tamaño contra una escritura “mal hecha” sin dictionary encoding, para que puedas mirar tamaños en disco.

In [52]:
import pyarrow.parquet as pq
import pyarrow as pa

case_dir = make_case_dir("case_02_categorical_encoding")
artifact_dir = case_dir / "artifact_good"
artifact_bad_dir = case_dir / "artifact_bad_manual"
artifact_bad_dir.mkdir(parents=True, exist_ok=True)

# Hago un dataset más grande para que el tamaño sea más visible.
trips = make_tripdataset(validated=True)
df_big = pd.concat([trips.data] * 3000, ignore_index=True)
trips_big = make_tripdataset(validated=True)
trips_big.data = df_big

report = write_trips(
    trips_big,
    artifact_dir,
    options=WriteTripsOptions(
        mode="overwrite",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
    ),
)

assert report.ok is True
good_parquet = artifact_dir / "trips.parquet"
assert good_parquet.exists()

# 1) Verificación fuerte de dictionary encoding
parquet_file = pq.ParquetFile(good_parquet)
try:
    names = parquet_file.schema_arrow.names
    for col_name in ["mode", "purpose"]:
        col_idx = names.index(col_name)
        encodings = {
            str(enc).upper()
            for enc in parquet_file.metadata.row_group(0).column(col_idx).encodings
        }
        assert any("DICTIONARY" in enc for enc in encodings), (
            f"{col_name} no quedó con dictionary encoding: {encodings}"
        )
finally:
    parquet_file.close()

# 2) Escritura "mala" para comparar tamaño (sin dictionary encoding)
df_bad = df_big.copy()
table_bad = pa.Table.from_pandas(df_bad, preserve_index=False)
pq.write_table(
    table_bad,
    artifact_bad_dir / "trips_bad_no_dictionary.parquet",
    compression="snappy",
    use_dictionary=False,
)

size_good = good_parquet.stat().st_size
size_bad = (artifact_bad_dir / "trips_bad_no_dictionary.parquet").stat().st_size

print("size_good =", size_good)
print("size_bad  =", size_bad)
print("ratio bad/good =", round(size_bad / size_good, 3) if size_good else None)

# No fuerzo una assert de tamaño porque depende del contenido y backend,
# pero en general debería verse que el "bad" no es más eficiente.
assert size_good > 0
assert size_bad > 0

show_ok("Test 6.3 - categóricos persistidos eficientemente")

size_good = 8882
size_bad  = 44025
ratio bad/good = 4.957
OK - Test 6.3 - categóricos persistidos eficientemente


### Test 6.4 - smoke test de read_trips happy path usando snapshot de schema

Qué prueba: lectura formal exitosa sin entregar schema en options, obligando a usar el snapshot del sidecar. 

In [54]:
case_dir = make_case_dir("case_03_read_from_snapshot")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)
write_report = write_trips(
    trips,
    artifact_dir,
    options=WriteTripsOptions(mode="error_if_exists"),
)

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True
assert len(read_report.issues) >= 1  # al menos READ.METADATA.VALIDATED_FORCED_FALSE
assert loaded.schema.version == trips.schema.version
assert read_report.parameters["schema"]["source"] == "metadata"

# Post-read: siempre no validado
assert loaded.metadata["is_validated"] is False

# Side effects de read en metadata
assert loaded.metadata["dataset_id"] == trips.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips.metadata["artifact_id"]
assert loaded.metadata["events"][-1]["op"] == "read_trips"

# Data esencialmente igual
pd.testing.assert_frame_equal(
    loaded.data.reset_index(drop=True),
    trips.data.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

display(read_report)
show_ok("Test 6.4 - read_trips happy path usando snapshot de schema")

OperationReport(ok=True, issues=[Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': False, 'action': 'force_unvalidated'})], summary={'n_rows': 3, 'n_columns': 12, 'path': 'tmp_smoke\\case_03_read_from_snapshot\\artifact', 'storage_format': 'parquet', 'schema_source': 'metadata', 'schema_mismatch': False, 'dataset_id': 'dset_test_001', 'dataset_id_status': 'loaded', 'artifact_id': 'art_f8de3e35-88c1-440d-88f5-e57235d83047', 'artifact_id_status': 'loaded'}, parameters={'path': 'tmp_smoke\\case_03_read_from_snapshot\\artifact', 'strict': False, 'keep_metadata': True, 'schema': {'source': 'metadata', 'version': '1.1'}})

OK - Test 6.4 - read_trips happy path usando snapshot de schema


### Test 6.5 - smoke test de read_trips happy path con schema explícito

Qué prueba: precedencia de options.schema sobre snapshot persistido.

In [56]:
case_dir = make_case_dir("case_04_read_with_schema_option")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=True)
write_trips(trips, artifact_dir, options=WriteTripsOptions(mode="error_if_exists"))

schema_override = make_trip_schema_minimal()
schema_override.version = "1.1-override"

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=schema_override,
        strict=False,
        keep_metadata=True,
    ),
)

assert read_report.ok is True
assert loaded.schema.version == "1.1-override"
assert read_report.parameters["schema"]["source"] == "options"

display(read_report)
show_ok("Test 6.5 - read_trips con schema explícito")

OperationReport(ok=True, issues=[Issue(level='warning', code='READ.SCHEMA.MISMATCH', message='El schema provisto por options no coincide con el snapshot persistido en el sidecar; se usará options.schema según precedencia.', field=None, source_field=None, row_count=None, details={'schema_source': 'options', 'schema_mismatch': True, 'version_options': '1.1-override', 'version_metadata': '1.1', 'required_diff': [], 'fields_diff_sample': [], 'fields_diff_total': 0, 'action': 'use_options_schema'}), Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': False, 'action': 'force_unvalidated'})], summary={'n_rows': 3, 'n_columns': 12, 'path': 'tmp_smoke\\case_04_read_with_schema_option\\artifact', 'storage_format': 'parquet', 'schema_source': 'options', 'schema_mismatch': True, 'dataset_id': 'dse

OK - Test 6.5 - read_trips con schema explícito


### Test 6.6 - smoke test fatal de write_trips por precondición no satisfecha

Qué prueba: require_validated=True con dataset no validado.

In [57]:
case_dir = make_case_dir("case_05_write_fatal_not_validated")
artifact_dir = case_dir / "artifact"

trips = make_tripdataset(validated=False)

raised = None
try:
    write_trips(
        trips,
        artifact_dir,
        options=WriteTripsOptions(
            mode="error_if_exists",
            require_validated=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValidationError)
assert not artifact_dir.exists()

display(raised)
show_ok("Test 6.6 - fatal de write_trips por dataset no validado")

ValidationError(message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', details={'require_validated': True, 'validated_flag': False, 'path': 'tmp_smoke\\case_05_write_fatal_not_validated\\artifact', 'action': 'abort'}, issue=Issue(level='error', code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", field=None, source_field=None, row_count=None, details={'require_validated': True, 'validated_flag': False, 'path': 'tmp_smoke\\case_05_write_fatal_not_validated\\artifact', 'action': 'abort'}), issues=(Issue(level='error', code='WRT.VALIDATION.REQUIRED_NOT_VALIDATED', message="write_trips requiere un dataset validado, pero metadata['is_validated']=False.", field=None, source_field=None, row_count=None, details={'require_validated': True, 'validated_flag': False, 'path': 'tmp_smoke\\case_05_write_fatal_not_validated\\artifact', 'act

OK - Test 6.6 - fatal de write_trips por dataset no validado


### Test 6.7 - smoke test fatal de read_trips por layout inválido

Qué prueba: artefacto con trips.parquet pero sin trips.metadata.json.

In [59]:
case_dir = make_case_dir("case_06_read_fatal_missing_sidecar")
artifact_dir = case_dir / "artifact"
artifact_dir.mkdir(parents=True, exist_ok=True)

# Creo un parquet mínimo manualmente, pero sin sidecar formal.
make_trip_df().to_parquet(
    artifact_dir / "trips.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

raised = None
try:
    read_trips(
        artifact_dir,
        options=ReadTripsOptions(strict=False, keep_metadata=True),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)

display(raised)
show_ok("Test 6.7 - fatal de read_trips por sidecar faltante")

ExportError(message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", code='READ.LAYOUT.MISSING_SIDECAR', details={'path': 'tmp_smoke\\case_06_read_fatal_missing_sidecar\\artifact', 'resolved_path': 'tmp_smoke\\case_06_read_fatal_missing_sidecar\\artifact', 'expected_file': 'trips.metadata.json', 'files_present_sample': ['trips.parquet'], 'files_present_total': 1, 'action': 'abort'}, issue=Issue(level='error', code='READ.LAYOUT.MISSING_SIDECAR', message="El artefacto formal de trips no contiene el sidecar obligatorio 'trips.metadata.json'.", field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke\\case_06_read_fatal_missing_sidecar\\artifact', 'resolved_path': 'tmp_smoke\\case_06_read_fatal_missing_sidecar\\artifact', 'expected_file': 'trips.metadata.json', 'files_present_sample': ['trips.parquet'], 'files_present_total': 1, 'action': 'abort'}), issues=(Issue(level='error', code='READ.LAYOUT.MISSING_SIDECAR', message="El artefa

OK - Test 6.7 - fatal de read_trips por sidecar faltante


### Test 6.8 - smoke test degradado de read_trips con recovery (strict=False)

Qué prueba: sidecar incompleto pero recuperable:

- falta schema_effective,
- artifact_id inválido,
- strict=False,
- la lectura retorna con warnings y fuerza is_validated=False.

In [61]:
case_dir = make_case_dir("case_07_read_degraded_recovery")
artifact_dir = case_dir / "artifact"

# Primero materializo un artefacto correcto
paths, payload = materialize_minimal_formal_artifact(artifact_dir)

# Luego lo degrado manualmente
payload_bad = copy.deepcopy(payload)
payload_bad.pop("schema_effective", None)
payload_bad["artifact_id"] = None
payload_bad["metadata"]["artifact_id"] = None

(artifact_dir / "trips.metadata.json").write_text(
    json.dumps(payload_bad, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

loaded, report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

codes = [i.code for i in report.issues]

assert report.ok is True
assert "READ.SCHEMA_EFFECTIVE.DEFAULTED" in codes
assert "READ.METADATA.ARTIFACT_ID_SET_NONE" in codes
assert "READ.METADATA.VALIDATED_FORCED_FALSE" in codes

assert loaded.metadata["is_validated"] is False
assert loaded.metadata["artifact_id"] is None

display(report.issues)
show_ok("Test 6.8 - read_trips degradado con recovery")

[Issue(level='warning', code='READ.SCHEMA_EFFECTIVE.DEFAULTED', message='schema_effective no está disponible o no es interpretable; se reconstruirá un estado efectivo vacío/default.', field=None, source_field=None, row_count=None, details={'reason': 'missing_schema_effective_snapshot', 'strict': False, 'action': 'default_empty_schema_effective'}),
 Issue(level='warning', code='READ.METADATA.ARTIFACT_ID_SET_NONE', message='El artifact_id persistido faltaba o era inválido; se dejará artifact_id=None en el dataset cargado.', field=None, source_field=None, row_count=None, details={'artifact_id': None, 'artifact_id_status': 'missing_or_invalid', 'previous_value': None, 'reason': 'missing_or_invalid_in_sidecar', 'action': 'set_none'}),
 Issue(level='info', code='READ.METADATA.VALIDATED_FORCED_FALSE', message="Se forzó metadata['is_validated']=False tras la lectura formal del artefacto de trips.", field=None, source_field=None, row_count=None, details={'previous_value': True, 'new_value': Fal

OK - Test 6.8 - read_trips degradado con recovery


### Test 6.9 - smoke test integrado de round-trip completo

Qué prueba: write + read juntos, comparando:

- data,
- schema,
- schema_effective,
- dataset_id,
- artifact_id,
- metadata y eventos,
- y la regla is_validated=False tras read.

In [62]:
case_dir = make_case_dir("case_08_roundtrip_full")
artifact_dir = case_dir / "artifact"

trips_original = make_tripdataset(validated=True)
data_original = trips_original.data.copy(deep=True)
schema_original = trips_original.schema
schema_effective_original = trips_original.schema_effective

write_report = write_trips(
    trips_original,
    artifact_dir,
    options=WriteTripsOptions(
        mode="error_if_exists",
        require_validated=True,
        storage_format="parquet",
        parquet_compression="snappy",
    ),
)

loaded, read_report = read_trips(
    artifact_dir,
    options=ReadTripsOptions(
        schema=None,
        strict=False,
        keep_metadata=True,
    ),
)

# Write y read felices
assert write_report.ok is True
assert read_report.ok is True

# Identidad
assert loaded.metadata["dataset_id"] == trips_original.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == trips_original.metadata["artifact_id"]

# Post-read: siempre no validado
assert loaded.metadata["is_validated"] is False

# Data esencialmente igual
pd.testing.assert_frame_equal(
    loaded.data.reset_index(drop=True),
    data_original.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

# Schema base y schema_effective equivalentes
assert loaded.schema.to_dict() == schema_original.to_dict()
assert loaded.schema_effective.to_dict() == schema_effective_original.to_dict()

# Metadata/eventos:
# - en memoria original tras write: debe existir write_trips
assert trips_original.metadata["events"][-1]["op"] == "write_trips"

# - en loaded tras read: debe estar write_trips persistido + read_trips append
ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_trips" in ops_loaded
assert ops_loaded[-1] == "read_trips"

print("ops_loaded =", ops_loaded)
display(write_report.summary)
display(read_report.summary)
show_ok("Test 6.9 - round-trip completo write/read")

ops_loaded = ['write_trips', 'read_trips']


{'n_rows': 3,
 'files_written': ['trips.parquet', 'trips.metadata.json'],
 'path': 'tmp_smoke\\case_08_roundtrip_full\\artifact',
 'dataset_id': 'dset_test_001',
 'artifact_id': 'art_9a04e55c-d219-4984-8232-1f83d3dbe610',
 'dataset_id_status': 'preserved',
 'storage_format': 'parquet'}

{'n_rows': 3,
 'n_columns': 12,
 'path': 'tmp_smoke\\case_08_roundtrip_full\\artifact',
 'storage_format': 'parquet',
 'schema_source': 'metadata',
 'schema_mismatch': False,
 'dataset_id': 'dset_test_001',
 'dataset_id_status': 'loaded',
 'artifact_id': 'art_9a04e55c-d219-4984-8232-1f83d3dbe610',
 'artifact_id_status': 'loaded'}

OK - Test 6.9 - round-trip completo write/read


### Grupo 6.10 - inspección rápida de artefactos persistidos

Qué prueba: no prueba lógica nueva; solo deja una forma rápida de inspeccionar lo escrito en disco al terminar los smoke tests.

In [63]:
print("Contenido de SMOKE_ROOT:")
for p in sorted(SMOKE_ROOT.rglob("*")):
    rel = p.relative_to(SMOKE_ROOT)
    kind = "DIR " if p.is_dir() else "FILE"
    size = "" if p.is_dir() else f" ({p.stat().st_size} bytes)"
    print(f"{kind:4} - {rel}{size}")

Contenido de SMOKE_ROOT:
DIR  - case_01_write_happy
DIR  - case_01_write_happy\artifact_write_happy
FILE - case_01_write_happy\artifact_write_happy\trips.metadata.json (5927 bytes)
FILE - case_01_write_happy\artifact_write_happy\trips.parquet (7655 bytes)
DIR  - case_02_categorical_encoding
DIR  - case_02_categorical_encoding\artifact_bad_manual
FILE - case_02_categorical_encoding\artifact_bad_manual\trips_bad_no_dictionary.parquet (44025 bytes)
DIR  - case_02_categorical_encoding\artifact_good
FILE - case_02_categorical_encoding\artifact_good\trips.metadata.json (5928 bytes)
FILE - case_02_categorical_encoding\artifact_good\trips.parquet (8882 bytes)
DIR  - case_03_read_from_snapshot
DIR  - case_03_read_from_snapshot\artifact
FILE - case_03_read_from_snapshot\artifact\trips.metadata.json (5917 bytes)
FILE - case_03_read_from_snapshot\artifact\trips.parquet (7655 bytes)
DIR  - case_04_read_with_schema_option
DIR  - case_04_read_with_schema_option\artifact
FILE - case_04_read_with_schem